# Nonparametric Testing

A compact Python-first guide to rank-based tests, assumption checks, p-values, and careful interpretation.

## 1. When and why use nonparametric methods?

Nonparametric tests are useful when a scientifically meaningful rank-based comparison is appropriate and strong parametric assumptions are doubtful. They can be valuable for ordinal data, skewed distributions, outliers, or small samples where assumptions cannot be justified.

They are not assumption-free: observations must still follow the required independent or paired design, and some median interpretations require similarly shaped distributions. They may also test a different hypothesis from the corresponding t-test. The choice should therefore follow the estimand and study design, not a mechanical normality-test rule.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
ALPHA = 0.05

## 2. Shapiro-Wilk normality test

$H_0$: the sample comes from a normal distribution. A small p-value is evidence against normality; a large p-value does **not** prove normality. With small samples the test may have low power, while with large samples it may detect minor deviations. Use it together with plots and subject-matter judgment.

In [ ]:
normal_sample = rng.normal(loc=0, scale=1, size=40)
skewed_sample = rng.exponential(scale=1, size=40)

shapiro_results = pd.DataFrame([
    {"sample": "Normal", "W": stats.shapiro(normal_sample).statistic, "p_value": stats.shapiro(normal_sample).pvalue},
    {"sample": "Exponential", "W": stats.shapiro(skewed_sample).statistic, "p_value": stats.shapiro(skewed_sample).pvalue},
])
shapiro_results.assign(reject_normality=lambda x: x.p_value < ALPHA).round(4)

## 3. Mann-Whitney U test: two independent groups

The test evaluates whether observations from one independent population tend to be larger than observations from the other. Under additional equal-shape assumptions, it is often interpreted as a location or median comparison. It is not simply a test of means.

In [ ]:
group_a = rng.lognormal(mean=0.0, sigma=0.7, size=35)
group_b = rng.lognormal(mean=0.35, sigma=0.7, size=30)
mw = stats.mannwhitneyu(group_a, group_b, alternative="two-sided", method="auto")

pd.Series({"U statistic": mw.statistic, "p-value": mw.pvalue, "reject H0": mw.pvalue < ALPHA})

## 4. Wilcoxon signed-rank test: paired observations

For paired data, apply the test to within-pair differences. The null hypothesis concerns symmetry around zero; the signed-rank test requires a symmetric distribution of differences. Zero differences and ties require attention.

In [ ]:
before = rng.normal(loc=70, scale=8, size=25)
after = before - rng.normal(loc=3, scale=4, size=25)
differences = after - before
wilcoxon_result = stats.wilcoxon(differences, alternative="two-sided", method="auto")

pd.Series({"W statistic": wilcoxon_result.statistic, "p-value": wilcoxon_result.pvalue, "reject H0": wilcoxon_result.pvalue < ALPHA})

## 5. Kruskal-Wallis test: three or more independent groups

The omnibus null hypothesis is that the group distributions are identical. A significant result indicates that at least one group differs, but it does not identify which groups. Post-hoc pairwise comparisons require multiplicity correction. Under equal-shape assumptions, the test is commonly interpreted as a location comparison.

In [ ]:
group_1 = rng.gamma(shape=2, scale=1.0, size=30)
group_2 = rng.gamma(shape=2, scale=1.3, size=30)
group_3 = rng.gamma(shape=2, scale=1.8, size=30)
kw = stats.kruskal(group_1, group_2, group_3)

pd.Series({"H statistic": kw.statistic, "p-value": kw.pvalue, "reject H0": kw.pvalue < ALPHA})

## 6. Parametric vs nonparametric comparison

The Welch t-test targets equality of population means and does not require equal variances. Mann-Whitney targets a rank/distributional comparison. Different p-values are not a contradiction: the tests answer different questions and use different information. Report an effect estimate and uncertainty alongside either test.

In [ ]:
welch = stats.ttest_ind(group_a, group_b, equal_var=False)
mann_whitney = stats.mannwhitneyu(group_a, group_b, alternative="two-sided", method="auto")

comparison = pd.DataFrame([
    {"test": "Welch t-test", "target": "difference in means", "statistic": welch.statistic, "p_value": welch.pvalue},
    {"test": "Mann-Whitney U", "target": "distribution/rank tendency", "statistic": mann_whitney.statistic, "p_value": mann_whitney.pvalue},
])
comparison.assign(reject_H0=lambda x: x.p_value < ALPHA).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].boxplot([group_a, group_b], tick_labels=["A", "B"], showmeans=True)
axes[0].set(title="Independent groups", ylabel="Observed value")
stats.probplot(differences, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot of paired differences")
fig.tight_layout()
plt.show()

## 7. Interpreting p-values

A p-value is the probability, **assuming the null hypothesis and model assumptions are true**, of obtaining a test statistic at least as extreme as the observed one. It is not the probability that $H_0$ is true, the size of an effect, or a measure of practical importance.

A concise report should include:

1. the design and test used;
2. the null and alternative hypotheses;
3. the statistic, sample sizes, and p-value;
4. an effect estimate and, where possible, a confidence interval;
5. a conclusion in the context of the research question.

## 8. Decision guide

| Design | Parametric target | Common rank-based alternative |
|---|---|---|
| Two independent groups | Means (Welch t-test) | Mann-Whitney U |
| Two paired measurements | Mean paired difference (paired t-test) | Wilcoxon signed-rank |
| Three or more independent groups | Means (ANOVA/Welch ANOVA) | Kruskal-Wallis |

Choose the analysis from the study design, estimand, measurement scale, and defensible assumptions. Do not choose a test solely because Shapiro-Wilk crossed the 0.05 threshold.